In [0]:
CREATE SCHEMA IF NOT EXISTS mini_project.bronze
  MANAGED LOCATION 's3://db-lakehouse-s3/lakehouse/bronze';

In [0]:
%python
import json


metadata_path = '/Volumes/mini_project/bronze/config_volume/ingestion_metadata.json'

with open(metadata_path, 'r') as f:
    ingestion_config = json.load(f)

for config in ingestion_config:
    source_file = config['source_file']
    target_table = config['target_table']
    s3_bucket = config['s3_bucket']
    s3_path = config['s3_path']
    
    full_s3_path = f"s3://{s3_bucket}/{s3_path}/{source_file}"
    
    create_table_sql = f"""
    CREATE OR REPLACE TABLE mini_project.bronze.{target_table} AS
    SELECT * FROM read_files(
      '{full_s3_path}',
      format => 'csv',
      header => true,
      inferSchema => true
    )
    """
    
    print(f"Creating table: mini_project.bronze.{target_table} from {source_file}...")
    spark.sql(create_table_sql)
    print(f"✓ Table mini_project.bronze.{target_table} created successfully\n")

print(f"✅ All {len(ingestion_config)} bronze tables created successfully!")